# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a walkthrough for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described via a Croissant schema JSON-LD file at the URL:

https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR² dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
print(f"Dataset loaded: {dataset.metadata.name}\n\n{dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id` values.

Croissant-compliant datasets define record sets uniquely identified by their `@id`. Let's enumerate accessible record sets and list their fields and field `@id`s for subsequent referencing.

In [ ]:
# List all record sets and their fields using @id
record_sets = list(dataset.list_record_sets())
if not record_sets:
    print("No record sets found in this dataset.")
else:
    print(f"Found Record Sets ({len(record_sets)}):\n")
    for rs in record_sets:
        print(f"- Record Set @id: {rs['@id']}")
        fields = rs.get('fields', [])
        if fields:
            print("  Fields:")
            for field in fields:
                field_id = field.get('@id', None)
                field_name = field.get('name', None)
                print(f"    - {field_id} (name: {field_name})")
        print()
if not record_sets:
    # Check if records() works without record_set id (single record set), and print a small sample
    sample_records = list(dataset.records())[:2]
    print(f"Sample records:\n{sample_records}")

## 3. Data Extraction
Load data from record sets into pandas DataFrames for analysis.

All entities are referenced by their `@id`. We'll demonstrate loading data for each record set found, else demonstrate generic extraction.

In [ ]:
# Extract data from record sets using @id; fallback if no record sets found
dataframes = {}

if record_sets:
    # There are record sets defined
    for rs in record_sets:
        record_set_id = rs['@id']
        try:
            records = list(dataset.records(record_set=record_set_id))
            if not records:
                print(f"No records found for record set {record_set_id}")
            else:
                df = pd.DataFrame(records)
                dataframes[record_set_id] = df
                print(f"Loaded {len(df)} records for record set: {record_set_id}")
                print(f"Sample columns: {df.columns.tolist()}")
                display(df.head(2))
        except Exception as e:
            print(f"Error loading record set {record_set_id}: {e}")
else:
    # No record sets; try generic records call
    try:
        records = list(dataset.records())
        df = pd.DataFrame(records)
        dataframes['main'] = df
        print(f"Loaded {len(df)} records.")
        print(f"Sample columns: {df.columns.tolist()}")
        display(df.head(2))
    except Exception as e:
        print(f"Error extracting records: {e}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on a numeric field, normalizing numeric fields, and grouping data.

We'll demonstrate filtering, normalization, and grouping based on accessible columns, referencing field or column `@id` as appropriate (using column names as proxies).

In [ ]:
# For demonstration, select numeric field if present and perform EDA
import numpy as np

if dataframes:
    # Use the first available DataFrame (fallback: main)
    record_set_id = next(iter(dataframes))
    df = dataframes[record_set_id]
    numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_candidates:
        # Try to convert any string columns that may be numeric
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col])
            except Exception:
                pass
        numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()

    if numeric_candidates:
        # Use the first numeric column and treat its name as field `@id`/proxy
        numeric_field_id = numeric_candidates[0]
        threshold = df[numeric_field_id].mean() if not np.isnan(df[numeric_field_id].mean()) else 0
        print(f"Numeric field identified for EDA: {numeric_field_id}\nUsing threshold: {threshold:.2f}")

        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records where {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Find a non-numeric field for grouping (categorical)
        group_candidates = df.select_dtypes(include=['object']).columns.tolist()
        group_field = None
        for g in group_candidates:
            if df[g].nunique() > 1 and df[g].nunique() < len(df) / 2:
                group_field = g
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(f"Grouped mean {numeric_field_id} by {group_field}:")
            display(grouped_df.head())
        else:
            print("No suitable categorical field found for grouping.")
    else:
        print("No numeric fields found for EDA in this record set.")
else:
    print("No dataframes available for EDA.")

## 5. Visualization
Let's visualize the distribution of a numeric field and, if available, compare it across categories of a grouping field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and 'numeric_field_id' in locals():
    # Histogram of the numeric variable
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id], kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()
    
    # If grouping field exists, boxplot
    if 'group_field' in locals() and group_field is not None:
        plt.figure(figsize=(10, 6))
        sns.boxplot(x=group_field, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.show()
else:
    print("Insufficient data for visualization.")

## 6. Conclusion
In this notebook, we demonstrated how to load and inspect a Croissant dataset package using `mlcroissant`, focusing on referencing all structures by `@id`. 

- **Dataset loaded:** Metadata and records with rich context for rangeland management studies in Kenya.
- **Record sets and fields:** Enumerated with `@id`, ready for fine-grained analysis.
- **Data extraction:** Loaded into pandas DataFrames for each record set (or generically if record sets are not explicitly defined).
- **EDA and visualization:** Applied basic filtering, normalization, grouping, and plotted numeric distributions, referencing all properties via their `@id` proxy/column.

For extended analysis, continue exploring relationships between additional dataset fields and entities using their `@id` for reproducibility and programmatic clarity.